# 02 · Length phenotypes of the mother cells

Each mother cell is turned into the vector of its **length** $L(t)$ on a regular time grid, all the mothers of
all the conditions are **clustered together**, drawn as **heatmaps** ordered by the dendrogram, and the notebook
reports **how often each phenotype appears in each condition**.

The length sees what the divisions (notebook 01) do not. In a heatmap of lengths:

- a mother that grows and divides normally gives a **saw-tooth**: it grows, halves at each division, grows again;
- a mother that stops growing (**arrest**) keeps a flat colour;
- a mother that keeps growing without dividing (**filament**) gets brighter and brighter.

An arrested mother and a filament have the same $N(t)$ (no divisions), but very different $L(t)$.

### How to use it

1. Fill in the cell **Your data** below: one line per folder of Excel files (`*_longitud_madres.xlsx`), with the
   areas you want and the condition they belong to.
2. Run all the cells (**Run › Run All Cells**).
3. The figures (PNG, 200 dpi, sized for slides), the tables (CSV) and the settings (`settings.json`) are saved in
   `results/02_length_phenotypes/<RUN_NAME>/`.

Nothing else has to be changed. The cells after **Your data** can be read but do not need to be edited.
Everything is done by functions of `mmtools` (the same ones a graphical interface would call:
`mm.run_length_phenotypes(EXPERIMENTS, ...)` does the whole notebook in one line).

## Your data

**`EXPERIMENTS`**: one `dict` per folder and condition.

| key | what to write | example |
|---|---|---|
| `folder` | folder with the `<area>-<date>_longitud_madres.xlsx` files (absolute, or relative to this notebook) | `'../November/6/Stacks/Excel'` |
| `condition` | name of the condition; several lines can share one condition (e.g. two experiments of the same treatment) | `'NIT'` |
| `areas` | `'all'`, a range `'1-10'`, or a list `'1-5, 8, 12-14'` | `'11-20'` |
| `flow_start` | time the flow started (`'HH:MM'`), which becomes time 0. If you leave it out (or `None`), time 0 is the **first image of that folder** | `'10:17'` |
| `name` | *(optional)* label of the experiment; by default the date in the file names | `'06/11'` |

**Other settings**

- `RUN_NAME`: name of the folder where this run's results are saved.
- `T_END`: end of the analysis window in minutes since time 0. `None` = the latest time that every condition still has images for.
- `STEP`: spacing of the time grid (min). The vector has one entry every `STEP` minutes.
- `N_PHENOTYPES`: number of phenotypes. `'auto'` = the one with the highest silhouette (section 3); or write a number.
- `CONDITION_COLORS`: *(optional)* your own colours, e.g. `{'LB': '#52514e', 'NIT': '#d9a21b'}`.

- `PAST_EXIT`: what to do with the mothers that grow **past the channel exit** (red cells in the Excel: they have
  no length, they are longer than the channel). `'exclude'` leaves them out of the clustering (they are the longest
  filaments, so the share of filaments is then a lower bound); `'cap'` gives them the channel length while they are
  past the exit.
- `CHANNEL_PX`: channel length in px for `PAST_EXIT = 'cap'`; `None` = the longest length measured in the data.
- `UM_PER_PX`: µm per pixel to draw the lengths in µm; `None` = keep pixels.
- `MAX_GAP`: up to this many consecutive missing grid points are filled by interpolation (see section 2).

In [ ]:
EXPERIMENTS = [
    dict(folder='../Analisis_NIT_GEN/data_gen_nit_bliss/LB_16_10_excel', areas='all',   condition='LB',      flow_start='10:45'),
    dict(folder='../November/6/Stacks/Excel',                         areas='31-40', condition='GEN',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='11-30', condition='NIT',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='1-10',  condition='NIT+GEN', flow_start='10:17'),
]

RUN_NAME = 'NIT_GEN_first_170min'
T_END = 170             # min since time 0, or None
STEP = 10               # min
N_PHENOTYPES = 'auto'   # 'auto' or a number
PAST_EXIT = 'exclude'   # 'exclude' or 'cap'
CHANNEL_PX = None       # px, only for PAST_EXIT = 'cap'; None = the longest length measured
UM_PER_PX = None        # e.g. 0.065; None = pixels
MAX_GAP = 2             # grid points
CONDITION_COLORS = {}   # optional, e.g. {'LB': '#52514e'}

## 1. Reading the Excel files

First the settings are checked (folders, areas, times); if something cannot be read, the notebook stops here and
says what. Then, for each folder, every `*_longitud_madres.xlsx` of the chosen areas is read: the time of each
image (row 1) and, for each channel, the mother's length and, in the row below, the daughter's length in the
image where a division is first seen.

In [ ]:
NOTEBOOK = '02_length_phenotypes'
SETTINGS = dict(step=STEP, t_end=T_END, n_phenotypes=N_PHENOTYPES, past_exit=PAST_EXIT,
                channel_px=CHANNEL_PX, um_per_px=UM_PER_PX, max_gap=MAX_GAP)

import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import mmtools as mm
from mmtools import plots as mp
warnings.filterwarnings('ignore')
mp.set_style()

problems = mm.check_entries(EXPERIMENTS)
if problems:
    raise ValueError('Please fix the cell "Your data":\n  - ' + '\n  - '.join(problems))

ds = mm.load_dataset(EXPERIMENTS, step=STEP, t_end=T_END, condition_colors=CONDITION_COLORS)
print('\n'.join(ds.messages))
OUT = Path('results') / NOTEBOOK / RUN_NAME
res = mm.AnalysisResult(NOTEBOOK, SETTINGS, ds.experiments, messages=list(ds.messages))

def show(name, fig):
    # show a figure and keep it to be saved at the end
    res.figures[name] = fig
    plt.show()

ds.summary

## 2. Each mother as a vector

At every grid point $t$ = `STEP`, 2·`STEP`, …, `T_END`, $L(t)$ is the length of the mother in the last image at or
before $t$, if that image is at most 12 min older (so a pause in the imaging leaves a gap instead of repeating an
old length).

Mothers that are left out (the table below says how many of each condition):

- **empty channel**: no cell in the first 3 images;
- **lost**: the top cell changed without a division, or the channel emptied for good, before `T_END`
  (as in notebook 01);
- **past exit**: past the channel exit at some grid point (only with `PAST_EXIT = 'exclude'`);
- **gaps**: more than `MAX_GAP` consecutive grid points without a reliable length (yellow cells of the Excel,
  defective images). Shorter gaps are filled by linear interpolation in time, or with the nearest value at the
  start or the end of the window (e.g. when an area has its first image just after the first grid point).

The clustering (next section) is computed here too; this can take a few seconds.

In [ ]:
P = mm.length_phenotypes(ds, n_phenotypes=N_PHENOTYPES, past_exit=PAST_EXIT, channel_px=CHANNEL_PX,
                         um_per_px=UM_PER_PX, max_gap=MAX_GAP)
P.counts

Some mothers of each condition (length rounded, one value per grid point):

In [ ]:
ex = pd.concat([P.ID[P.ID.condition == c].sample(min(3, (P.ID.condition == c).sum()), random_state=0) for c in ds.conditions])
print(f"{'condition':<12}{'exp.':<12}{'area':>5}{'ch.':>5}   L(t), t = {ds.grid[0]:g} … {ds.grid[-1]:g} min")
for i, r in ex.iterrows():
    print(f'{r.condition:<12}{r.experiment:<12}{r.area:>5}{r.channel:>5}   ' + ' '.join(f'{v:.0f}' for v in P.X[i]))

## 3. Clustering

- **Distance** between two mothers: root-mean-square difference of their vectors,
  $d(A,B) = \sqrt{\tfrac{1}{T}\sum_t \big(x_A(t) - x_B(t)\big)^2}$ ($T$ = number of grid points), in units of length.
  Because the distance is in absolute length, long filaments weigh much more than the saw-tooth of a dividing mother: the first split usually separates the long mothers from the rest, and the differences among normal-length mothers (dividing or arrested) are small in comparison.
- **Ward linkage** on that distance; the dendrogram orders the rows of the heatmaps.
- **Number of phenotypes $k$**, two criteria:
  - **silhouette**: how much closer each mother is to its own phenotype than to the nearest other one
    (1 = perfectly separated, 0 = on the border);
  - **stability**: the areas of each condition are resampled with replacement, the clustering is repeated and
    compared with the original (median adjusted Rand index, ARI; 1 = identical partition). The shaded band is the
    10th–90th percentile. A low 10th percentile means the partition depends on which areas are in the data.
- Phenotypes are numbered **P1, P2, …** from the shortest to the longest (by $L$ at the end of the window).

In [ ]:
show('k_selection', P.fig_k())
display(P.selection.round(3))
print(f'k = {P.k} phenotypes')
P.phenotype_summary().round(1)

## 4. Heatmaps

Rows = mothers, columns = time, colour = length of the mother (the longest 2 % saturate).

- **All the mothers together**: on the left, the dendrogram (coloured below the cut that gives the $k$
  phenotypes); on the right, the phenotype and the condition of each mother.
- **One heatmap per condition**, with the rows in the same order as in the global dendrogram.

In [ ]:
show('heatmap_dendrogram', P.fig_heatmap())
show('heatmaps_by_condition', P.fig_by_condition())

## 5. Mean trajectories

Mean $L(t)$ (± SD) of each phenotype, and of each condition.

In [ ]:
show('mean_by_phenotype', P.fig_mean_phenotype())
show('mean_by_condition', P.fig_mean_condition())

## 6. Fraction of each phenotype in each condition

Share of the mothers of each condition in each phenotype. The 95 % intervals come from resampling the **areas**
of each condition (mothers of the same area are not independent).

In [ ]:
show('fractions', P.fig_fractions())
show('fractions_CI', P.fig_fractions_ci())
P.fraction_table()

## 7. Lengths and divisions together

The mothers are also clustered by their divisions $N(t)$ as in notebook 01 (number of division phenotypes chosen
by the silhouette), and the two descriptions are compared:

- **Lengths in the order of the divisions**: the length heatmap of every mother used in the division clustering
  (so also those past the exit or with gaps, which are left out of the length clustering), with the rows in the
  order of the *division* dendrogram and the division phenotype on the left. Grey = no reliable length; past the
  exit = the top colour. Here the saw-tooth of the dividing mothers lines up, and among the non-dividing ones the
  arrested (dark) and the filaments (bright) are told apart.
- **Cross-table**: the mothers used in both clusterings, by division phenotype (rows) and length phenotype
  (columns). The adjusted Rand index (ARI) is 1 if the two partitions are the same and about 0 if they are
  unrelated (complementary descriptions).

In [ ]:
PN = mm.division_phenotypes(ds, n_boot=0)        # n_boot = 0: k from the silhouette only, no stability (faster)
show('lengths_in_division_order', mm.lengths_in_division_order(ds, PN, um_per_px=UM_PER_PX))
CT, ARI, fig = mm.compare_phenotypes(PN, P)
show('divisions_vs_length', fig)
print(f'{PN.k} division phenotypes;  ARI between the two partitions = {ARI:.2f}')
CT

## 8. Saving

In `results/02_length_phenotypes/<RUN_NAME>/`: the figures (numbered PNG) and

- `mothers.csv`: one row per mother used: condition, experiment, area, channel, phenotype and $L(t)$ (one column
  per grid point);
- `fractions.csv`, `k_selection.csv`, `phenotype_summary.csv`, `mothers_used.csv`, `divisions_vs_length.csv`,
  `experiments.csv`;
- `settings.json`: the experiments and settings of this run (`mm.load_config` reads it back);
- `messages.txt`.

In [ ]:
res.tables.update(P.tables())
res.tables['experiments'] = ds.summary
res.tables['divisions_vs_length'] = CT
res.messages.append(f'ARI divisions vs length = {ARI:.2f}')
res.save(OUT)
print('saved in', OUT.resolve())
sorted(p.name for p in OUT.iterdir())